# YOLOv8 + SimAM - Dikkat modulu - Yer seviyesi - catlak / yapisal hasar tespiti

**Katman:** Guclendirici

Catlak oznitelik temsilini guclendiriyor; literaturde raporlanmis kazanc. SimAM parametresiz dikkat -> `ultralytics` model YAML'ine eklenir.

Bizim yontem: 2x YOLOv8 (catlak.pt + bina.pt) + hibrit kip, guven esigi 0,20, gecikme 153-174 ms.  
Butun modeller `common.py` uzerinden AYNI Roboflow seti / AYNI bolme / AYNI metrik.  
Metrik: mAP@50, mAP@50-95, Kesinlik, Duyarlilik, F1, sinif bazli AP + FPS/Params/GFLOPs.

In [ ]:
# === COLAB KURULUM ===  (yerelde calistirinca atlanir)
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not os.path.exists('/content/QuakeMind'):
    !git clone --depth 1 https://github.com/UtBird/QuakeMind.git /content/QuakeMind
if IN_COLAB:
    %cd /content/QuakeMind/model_benchmarks/03_yer_seviyesi
    %pip install -q ultralytics roboflow
print('cwd:', os.getcwd())

## 1. Veri

In [ ]:
import sys; from pathlib import Path; sys.path.insert(0, str(Path.cwd()))
import common as C
C.CONFIG.update(dict(
    roboflow_api_key='BURAYA_ROBOFLOW_API_KEY',
    roboflow_workspace='workspace-slug',
    roboflow_project='project-slug',
    roboflow_version=1,
))
ds_dir = C.download_dataset('yolov8')
DATA_YAML = ds_dir / 'data.yaml'
print('veri:', DATA_YAML)

## 2. SimAM'li model YAML'i

`ultralytics/cfg/models/v8/yolov8.yaml` kopyalanir, backbone sonuna `[-1, 1, SimAM, []]` katmani eklenir (SimAM ultralytics >= 8.1'de kayitli). Asagida hazir bir kopya olusturuluyor.

In [ ]:
import ultralytics, yaml, shutil
base = Path(ultralytics.__file__).parent / 'cfg/models/v8/yolov8.yaml'
cfg = yaml.safe_load(open(base))
# backbone son SPPF'den sonra SimAM ekle
cfg['backbone'].insert(-1, [-1, 1, 'SimAM', []])
yaml.safe_dump(cfg, open('yolov8_simam.yaml', 'w'))
print('yazildi: yolov8_simam.yaml')

## 3. Egit

In [ ]:
from ultralytics import YOLO
MODEL_NAME = 'yolov8s_simam'
model = YOLO('yolov8_simam.yaml').load('yolov8s.pt')
model.train(data=str(DATA_YAML), epochs=C.CONFIG['epochs'], imgsz=C.CONFIG['img_size'],
            batch=C.CONFIG['batch_size'], seed=42, name=MODEL_NAME, exist_ok=True)
m = model.val(data=str(DATA_YAML), split='test')
prec, rec = float(m.box.mp), float(m.box.mr)
metrics = dict(map50=float(m.box.map50), map50_95=float(m.box.map), precision=prec, recall=rec,
               f1=2*prec*rec/(prec+rec+1e-9))
eff = C._yolo_efficiency(model, C.CONFIG['img_size'])
C.save_result(MODEL_NAME, metrics, eff, extra={'note': 'SimAM backbone'})
print(metrics)